# Validation: predicted power vs a real vLLM run

Everything so far has been a prediction. This notebook runs **the same traffic** through
real vLLM on a real GPU, samples board power with NVML, and scores the two against each
other.

**This one needs a GPU.** Ideally an A100 &mdash; see the honesty section below for why
that is not a preference.

---

## Read this before running anything

Three things decide whether the number at the end means anything. None of them are
fixable by trying harder in the notebook, so they are stated first rather than buried in
a caveats section at the bottom.

### 1. The LUT is an A100-40GB-PCIe at 900 MHz

EnergAIzer's tables were measured on one card at one clock. On a T4 or an L4 the
predicted watts describe **a different piece of silicon**, and disagreement tells you
nothing about the model. Even on an A100, if the clock is not pinned near 900 MHz you are
comparing two operating points.

The notebook checks the card and says which comparisons are still legitimate. On the
wrong hardware, **shape** agreement (does the trace move the same way?) is still
meaningful; **level** agreement is not.

### 2. vLLM eager is not HuggingFace eager

This is the big one, and it is a known gap, not a surprise. `enforce_eager=True` disables
**CUDA graphs**. It does not turn off PagedAttention. So vLLM still runs:

| | our kernel shapes | what vLLM actually runs |
|---|---|---|
| attention | four eager kernels per request per block | one paged/varlen kernel per block for the whole batch |
| KV write | not modelled at all | a paged cache write per step |
| elementwise | one kernel each, as HF traces them | several fused together |

The simulator's own diagnostics say decode attention is **86% of predicted energy**, and
that is precisely the part vLLM implements differently. So expect the prediction to run
**high**, and expect the gap to grow with batch size. A large disagreement here is the
experiment working &mdash; it puts a number on assumption 1 of the design doc, which has
been an unquantified caveat since the start.

### 3. GPT-2's context window is 1024 tokens

The simulator does not know that. `rewrite_dims` is arithmetic; it will price a
2048-token context happily and nothing about it is inconsistent. But GPT-2 cannot run it,
and vLLM will refuse. **Every earlier run in these notebooks used `max_tokens=2048`, so
they were extrapolating past the model's window.** The traffic here is generated inside
it instead.

---

**What a good result looks like.** Not zero error. A mean bias of 20-40% with high
`acf_r2` would mean the simulator gets the *shape* of the trace right and the *level*
wrong by a knowable factor &mdash; which is a usable model. Low `acf_r2` would be worse
news than a large bias, because a level offset can be calibrated and a wrong shape
cannot.

## 1 - What GPU did we get?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.sm,clocks.max.sm,power.limit --format=csv

## 2 - Install

vLLM is a large install and will restart the runtime on some Colab images. If the cell
below ends with a restart prompt, accept it and then **re-run from here** &mdash; nothing
above this point needs to persist.

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git'
DIR  = '/content/dynamic_shape_power_sim'
if not os.path.isdir(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, DIR], check=True)
else:
    subprocess.run(['git', '-C', DIR, 'pull', '--ff-only'], check=True)
if DIR not in sys.path:
    sys.path.insert(0, DIR)
os.chdir(DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'pandas', 'matplotlib', 'pynvml'], check=True)
print('installing vllm -- this is the slow one')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)

import dynshape
print('dynshape', dynshape.__version__)

## 3 - Is this the card the tables were measured on?

In [ ]:
from dynshape import PowerSampler, idle_baseline

sampler_probe = PowerSampler()
info = sampler_probe.device_info()
ok, why = sampler_probe.matches_lut_hardware(info)

for k, v in info.items():
    print(f'  {k:20s} {v}')
print()
print(('MATCH   ' if ok else 'MISMATCH') + ' -- ' + why)
print()
if ok:
    print('Both level and shape comparisons below are legitimate.')
else:
    print('Treat the LEVEL comparison as meaningless and read only the SHAPE')
    print('metrics (acf_r2, and nrmse after subtracting each side idle floor).')
    print('You can still learn whether the simulator moves the way the GPU moves.')

LUT_HARDWARE_MATCH = ok

### The idle floor, measured rather than assumed

The artifact's DVFS table says 47.35 W at 900 MHz for its A100. A different card, a
different driver, or anything else resident on the GPU moves that &mdash; and the floor
sits under **every** sample in the run, so an unmeasured floor is a systematic error in
the comparison rather than a rounding one.

Nothing should be on the GPU while this runs.

In [ ]:
idle = idle_baseline(seconds=8.0)
for k, v in idle.items():
    print(f'  {k:10s} {v:,.2f}' if isinstance(v, float) else f'  {k:10s} {v}')

IDLE_W = idle['median_w']
print()
print(f'measured idle floor      {IDLE_W:.2f} W')
print(f'artifact DVFS table @900 47.35 W')
print(f'difference               {IDLE_W - 47.35:+.2f} W  '
      f'({100*(IDLE_W - 47.35)/47.35:+.1f}%)')

## 4 - The traffic, generated once and used by both sides

This is the point of the exercise: **one workload, two engines**. The same arrival times,
the same prompt lengths, the same output lengths go to the simulator and to vLLM.

`build_replay_traffic` caps totals at 960 tokens so every request fits GPT-2's 1024-token
window with headroom. It raises rather than clamping if anything does not fit &mdash;
silently trimming would make the executed workload differ from the priced one, which is
the single thing this notebook exists to prevent.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dynshape import build_replay_traffic, to_spec, spec_summary, save_spec

pd.set_option('display.width', 170)

N_REQUESTS = 120
QPS        = 6.0        # keep this modest: GPT-2 on an A100 saturates fast
CV         = 2.0        # bursty, like real traffic

requests = build_replay_traffic(num_requests=N_REQUESTS, qps=QPS, cv=CV,
                                min_tokens=64, max_total_tokens=960,
                                theta=0.85, prefill_to_decode_ratio=4.0, seed=0)
spec = to_spec(requests)
save_spec(spec, 'replay_spec.json')

for k, v in spec_summary(spec).items():
    print(f'  {k:22s} {v:,.2f}' if isinstance(v, float) else f'  {k:22s} {v}')

## 5 - Side A: the simulator

Priced against EnergAIzer's measured tables if the LUT is available, otherwise the
roofline &mdash; and the notebook says which, because a validation against a SYNTHETIC
predictor measures something quite different from a validation against measured tables.

In [ ]:
from dynshape import (ShapeRewriter, build_predictor, EngineConfig,
                      SchedulerConfig, run_engine)

USE_LUT = False     # True -> clone the artifact and download the LUT (slow, see below)

rw = ShapeRewriter.from_dir('templates/gpt2')

if USE_LUT:
    from dynshape.energaizer import clone_artifact, download_lut, build_gee_predictor
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pyyaml', 'scipy', 'scikit-learn', 'cvxpy', 'opt-einsum', 'gdown'],
                   check=True)
    ART = clone_artifact('/content/energaizer')
    download_lut(ART)
    pred = build_gee_predictor(ART, use_precomputed_coeff=True, verbose=False)
else:
    pred = build_predictor(force_analytic=True)

cfg = EngineConfig(
    scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256,
                              block_size=16, max_tokens=1024),
    fuse_linear=True, record_kernels_until_ms=0.0, idle_w=IDLE_W)

trace = run_engine(requests, rw, pred, cfg, progress_every=200)
s = trace.summary()
print()
print(f"predictor      {s['backend']}")
print(f"measured model {s['is_measured_model']}")
for k in ('iterations', 'wall_time_s', 'duty_cycle', 'total_energy_j',
          'avg_power_w_wallclock', 'peak_iteration_power_w', 'output_tokens_per_s'):
    print(f'  {k:24s} {s[k]:,.4g}')

## 6 - Side B: real vLLM

`enforce_eager=True` disables CUDA graphs. It does **not** disable PagedAttention &mdash;
that is the gap described at the top, and it cannot be turned off.

Two details make the executed workload match the priced one token for token:

- prompts go in as **token ids**, not text. Tokenising a string of the right character
  count gives a token count that is only approximately right, and on the prefill axis
  that moves the largest GEMM in the model.
- outputs are pinned with `min_tokens == max_tokens` and `ignore_eos=True`. Otherwise
  GPT-2 stops whenever it likes and the decode phase &mdash; most of the trace &mdash;
  ends early and at a different length for every request.

In [ ]:
import asyncio, time
from dynshape import prompt_token_ids

from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine

engine_args = AsyncEngineArgs(
    model='gpt2',
    enforce_eager=True,           # no CUDA graphs (paged attention stays)
    dtype='bfloat16',
    max_model_len=1024,
    max_num_seqs=256,
    max_num_batched_tokens=2048,  # same token budget as the simulated scheduler
    gpu_memory_utilization=0.85,
    disable_log_requests=True,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)
print('vLLM engine up')

In [ ]:
# Warm up: the first request pays compilation and allocator costs that belong to
# nothing in the workload, and folding them into the trace would show up as a
# large phantom spike the simulator has no reason to predict.
async def warmup():
    sp = SamplingParams(max_tokens=16, min_tokens=16, ignore_eos=True, temperature=0.0)
    gen = engine.generate({'prompt_token_ids': prompt_token_ids(64, seed=999)},
                          sp, request_id='warmup')
    async for _ in gen:
        pass

await warmup()
print('warm')

In [ ]:
async def fire(req, t_zero, results):
    """Wait until this request's arrival time, then send it."""
    delay = req.arrival_s - (time.perf_counter() - t_zero)
    if delay > 0:
        await asyncio.sleep(delay)
    sent = time.perf_counter() - t_zero
    sp = SamplingParams(max_tokens=req.n_decode, min_tokens=req.n_decode,
                        ignore_eos=True, temperature=0.0)
    first = None
    n_out = 0
    gen = engine.generate({'prompt_token_ids': prompt_token_ids(req.n_prompt, seed=req.index)},
                          sp, request_id=f'r{req.index}')
    async for out in gen:
        if first is None and out.outputs and out.outputs[0].token_ids:
            first = time.perf_counter() - t_zero
        if out.outputs:
            n_out = len(out.outputs[0].token_ids)
    done = time.perf_counter() - t_zero
    results.append({'index': req.index, 'sent_s': sent, 'ttft_s': (first - sent) if first else None,
                    'e2e_s': done - sent, 'n_prompt': req.n_prompt,
                    'n_decode_asked': req.n_decode, 'n_decode_got': n_out})

async def replay(spec, t_zero):
    results = []
    await asyncio.gather(*(fire(r, t_zero, results) for r in spec))
    return results

sampler = PowerSampler(interval_s=0.01).start()
time.sleep(1.0)                        # a second of pre-roll idle, for the baseline
T_ZERO = time.perf_counter()
WORK_START_S = T_ZERO - sampler.t0     # where work begins, in the sampler clock

vllm_results = await replay(spec, T_ZERO)
WORK_END_S = time.perf_counter() - sampler.t0

time.sleep(1.0)                        # a second of post-roll
meas_t, meas_w = sampler.stop()

vdf = pd.DataFrame(vllm_results).sort_values('index')
print(f'{len(vdf)} requests replayed over {WORK_END_S - WORK_START_S:.1f} s')
print(f'{len(meas_t):,} NVML samples at {1000*(meas_t[-1]-meas_t[0])/len(meas_t):.1f} ms mean spacing')
print()
bad = vdf[vdf.n_decode_got != vdf.n_decode_asked]
if len(bad):
    print(f'WARNING: {len(bad)} requests produced a different number of tokens than asked.')
    print('The two engines then did different amounts of work and the comparison is void.')
    display(bad.head())
else:
    print('every request produced exactly the requested number of tokens --')
    print('both engines did the same work.')

## 7 - Line them up

Both sides binned onto the same 250 ms grid by the same box filter. The alignment is on
**the start of work**, not on sampling zero &mdash; the harness spends seconds loading a
model before the first request, and counting that as predicted idle would flatter the
prediction enormously.

In [ ]:
from dynshape import compare_to_trace

DT_S = 0.25

board = compare_to_trace(meas_t, meas_w, trace, dt_s=DT_S, measured_t0=WORK_START_S)
dyn   = compare_to_trace(meas_t, meas_w, trace, dt_s=DT_S, measured_t0=WORK_START_S,
                         subtract_idle=IDLE_W)

rows = ['measured_mean_w','predicted_mean_w','measured_peak_w','predicted_peak_w',
        'measured_energy_j','predicted_energy_j','energy_error_pct','mean_bias_pct',
        'rmse_w','nrmse_range','acf_r2','ks_agreement']
cmp = pd.DataFrame({'board power': [board[k] for k in rows],
                    f'dynamic only (idle {IDLE_W:.1f} W removed)': [dyn[k] for k in rows]},
                   index=rows)
display(cmp.round(3))
print(f"compared over {board['bins']} bins = {board['window_s']:.1f} s")

**Which column to read.** Board power is what a facility meter sees, and it is the
honest headline. But it is dominated by an idle floor that both sides agree on for free,
so it flatters the model. The right-hand column removes that floor from both sides and
asks the harder question: **does the simulator get the *dynamic* power right?**

`mean_bias_pct` is signed on purpose. Systematic over-prediction is the kind of error
that survives aggregation, and it is also the kind that can be calibrated out &mdash; so
a large signed bias with a high `acf_r2` is a much better result than the reverse.

In [ ]:
from dynshape.engine_plot import add_alpha_gradient_line, STANFORD_RED

ser = board['_series']
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax = axes[0]
ax.plot(ser['t_s'], ser['measured'], lw=1.0, color='black', label='measured (NVML)')
ax.plot(ser['t_s'], ser['predicted'], lw=1.2, color=STANFORD_RED, label='predicted')
ax.axhline(IDLE_W, color='#888', ls=':', lw=1, label=f'measured idle {IDLE_W:.0f} W')
ax.set_ylabel('board GPU power (W)'); ax.set_ylim(bottom=0)
ax.set_title(f'Board power, {DT_S*1000:.0f} ms bins  ·  '
             f"bias {board['mean_bias_pct']:+.1f}%  ·  acf R2 {board['acf_r2']:.3f}")
ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=.25)

ax = axes[1]
d = dyn['_series']
ax.plot(d['t_s'], d['measured'], lw=1.0, color='black', label='measured − idle')
ax.plot(d['t_s'], d['predicted'], lw=1.2, color=STANFORD_RED, label='predicted − idle')
ax.set_xlabel('time from first request (s)'); ax.set_ylabel('dynamic power (W)')
ax.set_title(f"Dynamic power only  ·  bias {dyn['mean_bias_pct']:+.1f}%  ·  "
             f"acf R2 {dyn['acf_r2']:.3f}")
ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=.25)

fig.tight_layout(); plt.show()

In [ ]:
# Predicted against measured, bin by bin. The diagonal is agreement; the slope
# is the calibration factor a level offset would need.
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for a, src, name in ((ax[0], board, 'board'), (ax[1], dyn, 'dynamic only')):
    s = src['_series']
    a.scatter(s['measured'], s['predicted'], s=14, alpha=.55, color='#2a78d6', edgecolors='none')
    lo = float(min(s['measured'].min(), s['predicted'].min()))
    hi = float(max(s['measured'].max(), s['predicted'].max()))
    a.plot([lo, hi], [lo, hi], 'k--', lw=.9, label='agreement')
    slope = float(np.polyfit(s['measured'], s['predicted'], 1)[0]) if len(s['measured']) > 2 else float('nan')
    a.plot([lo, hi], [lo*slope, hi*slope], color=STANFORD_RED, lw=1.1, label=f'slope {slope:.2f}')
    a.set_xlabel(f'measured {name} (W)'); a.set_ylabel(f'predicted {name} (W)')
    a.set_title(name); a.legend(fontsize=8); a.grid(alpha=.25)
fig.tight_layout(); plt.show()

print('A slope near 1 with tight scatter is agreement. A consistent slope away from 1')
print('is a calibration factor -- usable. Wide scatter at any slope is not.')

## 8 - Latency, as a second opinion

Power is what we set out to predict, but the simulator also predicts timing, and timing
is measured here for free. It is a useful cross-check because the two failures look
different: if predicted *time* is right and predicted *power* is wrong, the kernel model
is off. If time is wrong too, the kernel list itself is wrong &mdash; which is what the
eager-vs-paged attention gap would do.

In [ ]:
sim = pd.DataFrame(trace.request_metrics())
m_ttft = vdf.ttft_s.dropna(); s_ttft = sim.ttft_s.dropna()
m_e2e  = vdf.e2e_s.dropna();  s_e2e  = sim.e2e_s.dropna()

lat = pd.DataFrame({
    'measured (vLLM)': [m_ttft.median(), m_ttft.quantile(.99), m_e2e.median(),
                        WORK_END_S - WORK_START_S],
    'predicted':       [s_ttft.median(), s_ttft.quantile(.99), s_e2e.median(),
                        trace.total_time_ms/1000],
}, index=['TTFT p50 (s)', 'TTFT p99 (s)', 'E2E p50 (s)', 'wall clock (s)'])
lat['ratio'] = lat['predicted'] / lat['measured (vLLM)']
display(lat.round(4))

print()
print('The wall-clock row is the one to look at first. Both engines were given the same')
print('arrivals, so if neither is saturated both should finish at about the same time;')
print('a predicted wall clock much longer than measured means the simulator thinks the')
print('work is heavier than it is -- which is exactly what per-request eager attention')
print('would do against vLLM one fused paged kernel.')

## 9 - Export

In [ ]:
out = pd.DataFrame({'t_s': board['_series']['t_s'],
                    'measured_w': board['_series']['measured'],
                    'predicted_w': board['_series']['predicted']})
out.to_csv('validation_250ms.csv', index=False)
vdf.to_csv('vllm_requests.csv', index=False)
pd.DataFrame([{k: v for k, v in board.items() if not k.startswith('_')}]).to_csv(
    'validation_metrics.csv', index=False)
print('validation_250ms.csv', out.shape)
print('vllm_requests.csv', vdf.shape)
print('validation_metrics.csv')

try:
    from google.colab import files
    for f_ in ('validation_250ms.csv', 'vllm_requests.csv', 'validation_metrics.csv'):
        files.download(f_)
except Exception:
    print('(not on Colab)')

## What this run can and cannot tell you

**It can tell you** whether the simulator's power trace moves the way a real GPU's does
under identical traffic, and by what factor the level is off. Those are the two things
that were previously unknown.

**It cannot tell you** that the kernel model is right, because four differences are
confounded in a single number:

1. **eager vs paged attention** &mdash; the big one, and the one the simulator's own
   diagnostics say carries 86% of predicted energy.
2. **the missing KV-write kernel** &mdash; not modelled at all here; roughly 0.014% of a
   vLLM decode step, so small, but not zero.
3. **fused elementwise ops** &mdash; vLLM fuses several of what HuggingFace traces
   separately, so the simulator counts more launches than actually happen.
4. **additivity** &mdash; kernel costs are summed with no overlap or memory contention.
   Untested, and a mixed batch stresses it hardest.

Separating those needs a per-kernel profile (Nsight Compute) against the same workload,
not a board-power trace. This run bounds the total; it does not attribute it.

**One honest reading of a bad result:** if the level is far off but `acf_r2` is high, the
simulator is a good *relative* model and a poor absolute one. For questions of the form
"does this scheduler change reduce peak power?" that is enough. For "how many watts will
this rack draw?" it is not.